In [1]:
import pandas as pd

### Part A : Data Spine 

In [3]:
store = pd.read_csv("store_transactions_sparse.csv", parse_dates=["date"])

print("Rows:", len(store))

Rows: 405


In [4]:
# 1. Store transactions — Daily data

# Number of calendar days between min and max date
calendar_days = pd.date_range(
    store["date"].min(),
    store["date"].max(),
    freq="D"
)

print("Calendar days:", len(calendar_days))
print("Missing days:", len(calendar_days) - len(store))

Calendar days: 424
Missing days: 19


In [5]:
# 2. Create the complete daily date spine

store = store.set_index("date")

store_spine = pd.date_range(
    store.index.min(),
    store.index.max(),
    freq="D"
)

store = store.reindex(store_spine)
store.index.name = "date"

In [6]:
# 3 & 4. Fill missing days and track store closures

# True = date was missing from the original data
store["was_closed"] = store["transactions"].isna()

# Missing sales mean the store was closed → use 0
store[["transactions", "revenue"]] = (
    store[["transactions", "revenue"]].fillna(0)
)

print(store.head())

            transactions   revenue  was_closed
date                                          
2023-01-02         788.0  15680.68       False
2023-01-03         799.0  15184.39       False
2023-01-04         722.0  10945.52       False
2023-01-05         862.0  14243.50       False
2023-01-06         909.0  14807.19       False


In [7]:
# 5. Energy usage — Hourly data

energy = pd.read_csv(
    "energy_usage_hourly.csv",
    parse_dates=["timestamp"]
)

energy_spine = pd.date_range(
    energy["timestamp"].min(),
    energy["timestamp"].max(),
    freq="h"
)

energy = (
    energy.set_index("timestamp")
          .reindex(energy_spine)
)

energy.index.name = "timestamp"

In [8]:
# Missing hours are sensor downtime, NOT zero energy usage.
# Interpolate between surrounding measurements.
energy["kwh"] = energy["kwh"].interpolate(method="linear")

print(energy.head())

                       kwh
timestamp                 
2024-03-01 00:00:00  0.931
2024-03-01 01:00:00  0.740
2024-03-01 02:00:00  1.289
2024-03-01 03:00:00  0.956
2024-03-01 04:00:00  1.259


### PART B — LAG AND LEAD

In [9]:
# 6. Lag features
store["lag_1"] = store["revenue"].shift(1)   # yesterday
store["lag_7"] = store["revenue"].shift(7)   # same day last week

In [10]:
# 7. Lead feature
store["lead_1"] = store["revenue"].shift(-1)  # tomorrow

In [11]:
# 8. Day-over-day change
store["revenue_change"] = store["revenue"] - store["lag_1"]

store["revenue_pct_change"] = store["revenue"].pct_change() * 100

print(store[[
    "revenue", "lag_1", "revenue_change", "revenue_pct_change"
]].head(10))

             revenue     lag_1  revenue_change  revenue_pct_change
date                                                              
2023-01-02  15680.68       NaN             NaN                 NaN
2023-01-03  15184.39  15680.68         -496.29           -3.164978
2023-01-04  10945.52  15184.39        -4238.87          -27.915972
2023-01-05  14243.50  10945.52         3297.98           30.130866
2023-01-06  14807.19  14243.50          563.69            3.957524
2023-01-07  17291.33  14807.19         2484.14           16.776579
2023-01-08  14855.74  17291.33        -2435.59          -14.085614
2023-01-09  12292.29  14855.74        -2563.45          -17.255620
2023-01-10  13126.67  12292.29          834.38            6.787832
2023-01-11  13787.06  13126.67          660.39            5.030903


In [15]:
# 10. Record day
store["lag_14"] = store["revenue"].shift(14)

store["record_day"] = (
    (store["revenue"] > store["lag_7"]) &
    (store["revenue"] > store["lag_14"])
)

print(store.head(5))

            transactions   revenue  was_closed     lag_1  lag_7    lead_1  \
date                                                                        
2023-01-02         788.0  15680.68       False       NaN    NaN  15184.39   
2023-01-03         799.0  15184.39       False  15680.68    NaN  10945.52   
2023-01-04         722.0  10945.52       False  15184.39    NaN  14243.50   
2023-01-05         862.0  14243.50       False  10945.52    NaN  14807.19   
2023-01-06         909.0  14807.19       False  14243.50    NaN  17291.33   

            revenue_change  revenue_pct_change  lag_14  record_day  
date                                                                
2023-01-02             NaN                 NaN     NaN       False  
2023-01-03         -496.29           -3.164978     NaN       False  
2023-01-04        -4238.87          -27.915972     NaN       False  
2023-01-05         3297.98           30.130866     NaN       False  
2023-01-06          563.69            3.957524

### PART C — ROLLING WINDOWS

In [16]:
# 11. 7-day trailing rolling mean
store["rolling_7_mean"] = store["revenue"].rolling(7).mean()

In [17]:
# 12. 7-day centered rolling mean
store["centered_7_mean"] = store["revenue"].rolling(7, center=True).mean()

print(store[[
    "revenue", "rolling_7_mean", "centered_7_mean"
]].head(12))

             revenue  rolling_7_mean  centered_7_mean
date                                                 
2023-01-02  15680.68             NaN              NaN
2023-01-03  15184.39             NaN              NaN
2023-01-04  10945.52             NaN              NaN
2023-01-05  14243.50             NaN     14715.478571
2023-01-06  14807.19             NaN     14231.422857
2023-01-07  17291.33             NaN     13937.462857
2023-01-08  14855.74    14715.478571     14343.397143
2023-01-09  12292.29    14231.422857     14575.475714
2023-01-10  13126.67    13937.462857     14857.512857
2023-01-11  13787.06    14343.397143     15530.970000
2023-01-12  15868.05    14575.475714     15435.311429
2023-01-13  16781.45    14857.512857     15403.775714


In [18]:
# 13. Rolling standard deviation + unusual days
store["rolling_7_std"] = store["revenue"].rolling(7).std()

store["unusual"] = (
    (store["revenue"] - store["rolling_7_mean"]).abs()
    > 2 * store["rolling_7_std"]
)

print("Unusual days:", store["unusual"].sum())

Unusual days: 14


In [19]:
# 14. Rolling mean with min_periods=1
store["rolling_7_mean_min1"] = (
    store["revenue"].rolling(7, min_periods=1).mean()
)

In [22]:
# 15. Expanding mean
store["expanding_mean"] = store["revenue"].expanding().mean()

print(store.head(5))

            transactions   revenue  was_closed     lag_1  lag_7    lead_1  \
date                                                                        
2023-01-02         788.0  15680.68       False       NaN    NaN  15184.39   
2023-01-03         799.0  15184.39       False  15680.68    NaN  10945.52   
2023-01-04         722.0  10945.52       False  15184.39    NaN  14243.50   
2023-01-05         862.0  14243.50       False  10945.52    NaN  14807.19   
2023-01-06         909.0  14807.19       False  14243.50    NaN  17291.33   

            revenue_change  revenue_pct_change  lag_14  record_day  \
date                                                                 
2023-01-02             NaN                 NaN     NaN       False   
2023-01-03         -496.29           -3.164978     NaN       False   
2023-01-04        -4238.87          -27.915972     NaN       False   
2023-01-05         3297.98           30.130866     NaN       False   
2023-01-06          563.69            3.

### PART D — RESAMPLING & AGGREGATION

In [23]:
# 16. Weekly totals
weekly_store = store.resample("W").agg({
    "transactions": "sum",
    "revenue": "sum"
})

In [24]:
# 17. Monthly totals + average transaction value
monthly_store = store.resample("MS").agg({
    "transactions": "sum",
    "revenue": "sum"
})

monthly_store["avg_transaction_value"] = (
    monthly_store["revenue"] / monthly_store["transactions"]
)

print(monthly_store)

            transactions    revenue  avg_transaction_value
date                                                      
2023-01-01       26900.0  474183.38              17.627635
2023-02-01       25872.0  461513.98              17.838357
2023-03-01       29159.0  524963.33              18.003475
2023-04-01       28423.0  518893.22              18.256103
2023-05-01       28360.0  522260.61              18.415395
2023-06-01       29856.0  548136.22              18.359332
2023-07-01       30326.0  552375.99              18.214601
2023-08-01       28551.0  505388.02              17.701237
2023-09-01       29596.0  540126.82              18.249994
2023-10-01       31848.0  572320.81              17.970385
2023-11-01       29989.0  545313.07              18.183770
2023-12-01       33710.0  587578.85              17.430402
2024-01-01       27943.0  511081.51              18.290145
2024-02-01       32925.0  597527.78              18.148148


In [25]:
# 18. Energy: hourly → daily → weekly
daily_energy = energy.resample("D")["kwh"].sum()
weekly_energy = energy.resample("W")["kwh"].sum()

print(daily_energy.head())
print(weekly_energy.head())

# Upsample daily totals back to hourly
hourly_energy = daily_energy.resample("h").asfreq()

print(hourly_energy.head(5))

timestamp
2024-03-01    62.069
2024-03-02    51.795
2024-03-03    50.721
2024-03-04    62.880
2024-03-05    62.527
Freq: D, Name: kwh, dtype: float64
timestamp
2024-03-03    164.585
2024-03-10    415.150
2024-03-17    414.789
2024-03-24    411.572
2024-03-31    360.125
Freq: W-SUN, Name: kwh, dtype: float64
timestamp
2024-03-01 00:00:00    62.069
2024-03-01 01:00:00       NaN
2024-03-01 02:00:00       NaN
2024-03-01 03:00:00       NaN
2024-03-01 04:00:00       NaN
Freq: h, Name: kwh, dtype: float64


In [26]:
# 19. Average revenue by weekday
weekday_revenue = (
    store.groupby(store.index.dayofweek)["revenue"]
         .mean()
         .reindex(range(7))
         .rename(index={
             0: "Monday", 1: "Tuesday", 2: "Wednesday",
             3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday"
         })
         .reset_index()
)

weekday_revenue.columns = ["weekday", "average_revenue"]

print(weekday_revenue)

     weekday  average_revenue
0     Monday     15573.160328
1    Tuesday     16036.700164
2  Wednesday     16809.885410
3   Thursday     16720.768033
4     Friday     17900.058167
5   Saturday     20426.670333
6     Sunday     19808.142167


In [27]:
print("Weekly rows:", len(weekly_store))
print(weekly_store.head())

# 20. Monthly revenue — two methods
monthly_direct = store.resample("MS")["revenue"].sum()

monthly_from_weekly = (
    store.resample("W")["revenue"]
         .sum()
         .resample("MS")
         .sum()
)

comparison = pd.DataFrame({
    "direct": monthly_direct,
    "from_weekly": monthly_from_weekly
})

comparison["match"] = comparison["direct"].eq(comparison["from_weekly"])

print(comparison)

Weekly rows: 61
            transactions    revenue
date                               
2023-01-08        6117.0  103008.35
2023-01-15        6233.0  108047.18
2023-01-22        6413.0  117280.35
2023-01-29        6516.0  118372.03
2023-02-05        6340.0  112646.62
               direct  from_weekly  match
date                                     
2023-01-01  474183.38    446707.91  False
2023-02-01  461513.98    459072.97  False
2023-03-01  524963.33    479508.02  False
2023-04-01  518893.22    594265.01  False
2023-05-01  522260.61    473253.75  False
2023-06-01  548136.22    511466.69  False
2023-07-01  552375.99    624196.50  False
2023-08-01  505388.02    447823.03  False
2023-09-01  540126.82    493848.01  False
2023-10-01  572320.81    654735.96  False
2023-11-01  545313.07    507853.22  False
2023-12-01  587578.85    660323.23  False
2024-01-01  511081.51    450300.78  False
2024-02-01  597527.78    584718.71  False
2024-03-01        NaN     73589.80  False


### PART E — MINI INTEGRATION CHALLENGE

In [28]:
# 1. Load data
store = pd.read_csv(
    "store_transactions_sparse.csv",
    parse_dates=["date"]
)

In [29]:
# 2. Build complete daily spine
spine = pd.date_range(
    store["date"].min(),
    store["date"].max(),
    freq="D"
)

store = store.set_index("date").reindex(spine)
store.index.name = "date"

In [30]:
# 3. Fill missing days
store["was_closed"] = store["transactions"].isna()

store[["transactions", "revenue"]] = (
    store[["transactions", "revenue"]].fillna(0)
)

In [31]:
# 4. 7-day rolling average of daily revenue
store["rolling_7_revenue"] = (
    store["revenue"].rolling(7).mean()
)

In [32]:
# 5. Create weekly summary
weekly = store.resample("W").agg(
    total_revenue=("revenue", "sum"),
    total_transactions=("transactions", "sum"),
    avg_daily_revenue=("revenue", "mean")
)

In [33]:
# 6. 4-week rolling average
weekly["rolling_4_week_revenue"] = (
    weekly["total_revenue"].rolling(4).mean()
)

In [ ]:
# 7. Week-over-week percentage change
weekly["wow_pct_change"] = (
    weekly["total_revenue"].pct_change() * 100
)


In [ ]:
# 8. Final tidy table
weekly